In [1]:
import hashlib
import json
import random
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

from datasets.augmentation import build_image_transform, build_batch_transform
from datasets.imagenette import download_imagenette_dataset
from models.vision_transformer import (
    build_vision_transformer_single_classifier,
)
from pipeline.evaluate import evaluate_model
from pipeline.train import train_model


SEED = 0
SPLIT_SEED = 0
EPOCHS = 100
WARMUP_EPOCHS = 5
BATCH_SIZE = 32
PEAK_LR = 5e-4
WEIGHT_DECAY = 0.01
IMAGE_SIZE = 224
RESULTS_DIR = Path("artifacts/part3/") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
DATA_DIR = Path("data")
MODEL_CONFIG = dict(image_size=224, patch_size=16, hidden_dim=192, num_heads=4,
                    num_layers=6, mlp_ratio=4, num_classes=10, dropout=0.1)
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")

device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")

def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=lambda value: value.tolist()) + "\n")

print(f"Device: {device}; results: {RESULTS_DIR}")

Device: mps; results: artifacts/part3/20261117_023324_UTC


In [2]:
pool = download_imagenette_dataset(DATA_DIR, split="train", image_size=IMAGE_SIZE)
labels = np.array([label for _, label in pool._samples])
train_indices, val_indices = train_test_split(np.arange(len(pool)), test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
train_data = Subset(pool, train_indices.tolist())
validation_data = Subset(pool, val_indices.tolist())
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
np.savez(RESULTS_DIR / "split_indices.npz", train=train_indices, validation=val_indices)
save_json(RESULTS_DIR / "dataset_manifest.json", {
    "dataset": "Imagenette", "distribution": "320px", "official_split": "train",
    "split_seed": SPLIT_SEED, "class_to_idx": pool.class_to_idx,
    "ordered_samples": [
        {"image": str(Path(path).relative_to(Path(pool._image_root))), "label": int(label)}
        for path, label in pool._samples
    ],
})
print(f"Training: {len(train_data)}; internal validation: {len(validation_data)}")

Training: 7575; internal validation: 1894


In [15]:
def window_partition(x, window_size):
    """
    Input:
        x: [B, H, W, C]
    Output:
        windows: [B * (H // M) * (W // M), M, M, C]
    """
    b, c, h, w = x.shape

    print(h%window_size)
    if h%window_size!=0 or w%window_size!=0:
        raise ValueError("height and width must be divisible by the window size")
    
    new_b = b*(h//window_size)*(w//window_size)
    return x.view(new_b, c, window_size, window_size)

def window_reverse(windows, window_size, height, width):
    """
    Input:
        windows: [B * (H // M) * (W // M), M, M, C]
    Output:
        x: [B, H, W, C]
    """
    b, c, h, w = windows.shape

    if height%window_size!=0 or width%window_size!=0:
        raise ValueError("height and width must be divisible by the window size")
    
    new_b = b//(height//window_size)//(width//window_size)
    return windows.view(new_b, c, height, width)



In [ ]:
from PIL import Image
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

image = Image.open("data/image_8765cd2c.jpg").resize((224,224))
transform = ToTensor()
image_tensor = transform(image).unsqueeze(dim=0)

print(f"image shape: {image_tensor.shape}")
windows = window_partition(image_tensor, 4)
print(f"windows shape: {windows.shape}")

image shape: torch.Size([1, 3, 224, 224])
0
windows shape: torch.Size([3136, 3, 4, 4])


RuntimeError: Boolean value of Tensor with more than one value is ambiguous